# Statistical API — zonal NDVI stats over a polygon

`api='statistical'` + `geometry=` computes zonal statistics over a polygon and writes a tidy CSV (min/max/mean/stDev + percentiles per interval). Uses the `sentinel-2-l2a-ndvi-stats` recipe (its evalscript emits the required `dataMask` band).

## Credentials guard

In [ ]:
import os
from pathlib import Path


def has_sh_credentials() -> bool:
    """Whether Sentinel Hub OAuth client-credentials are available."""
    return bool(os.environ.get("SENTINELHUB_CLIENT_ID") and os.environ.get("SENTINELHUB_CLIENT_SECRET"))


def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    Sentinel Hub credentials are configured (so the docs build never needs
    secrets). Set SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET to run the cell for real.
    """
    if not has_sh_credentials():
        print(
            "No Sentinel Hub credentials found - skipping the live download.\n"
            "Mint an OAuth client_credentials pair in the CDSE Dashboard and set\n"
            "SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET (see the Authentication page)."
        )
        return []
    results = facade.download(**download_kwargs)
    for item in results:
        print(item)
    return results


## Build + compute

In [ ]:
from earthlens import EarthLens

polygon = {
    'type': 'Polygon',
    'coordinates': [[[14.24, 40.80], [14.27, 40.80],
                     [14.27, 40.83], [14.24, 40.83], [14.24, 40.80]]],
}

facade = EarthLens(
    data_source='sentinel-hub',
    dataset='sentinel-2-l2a-ndvi-stats', variables=[],
    start='2020-06-01', end='2020-06-30',
    aoi=[14.24, 40.80, 14.27, 40.83],
    path='data/sh-stats', resolution=20,
    api='statistical', geometry=polygon,
)
tables = run_or_skip(facade)
tables

## Inspect the table (when it ran)

In [ ]:
import pandas as pd

if tables:
    frame = pd.read_csv(tables[0])
    display(frame.head())
else:
    print('No table written (no credentials) - see the cell above.')